# **H5N1 Outbreak Data: Multi-Source Ground Truth Generator**

## **Introduction**

Accurate outbreak modeling requires a high-quality, time-aligned ground truth that reflects real-world disease activity. However, H5N1 cases are reported inconsistently across human health, veterinary, media, and biosurveillance systems. No single source provides complete coverage. To address this challenge, this notebook constructs a **unified, daily, country-level H5N1 outbreak ground truth dataset** by integrating multiple independent data streams.

The pipeline aggregates information from global media reporting (GDELT), expert-curated alerts (ProMED), authoritative veterinary reporting (FAO EMPRES-i), and human case notifications (WHO Disease Outbreak News). Together, these sources provide a robust, multi-modal representation of both **animal** and **human** H5N1 activity. This consolidated dataset is used as the foundation for downstream early-warning modeling, cross-correlation analysis, and digital epidemiology research.

---

## **Objectives**

### **1. Consolidate multiple outbreak information sources**

* Extract digital outbreak signals from **GDELT (global media)**
* Parse expert alerts from **ProMED**
* Retrieve confirmed **animal cases** from **FAO EMPRES-i**
* Scrape human outbreak notifications from **WHO DON**

### **2. Normalize and unify heterogeneous date formats and metadata**

* Implement a robust datetime parser for irregular GDELT formats
* Standardize country identifiers and missing metadata
* Harmonize all sources into a common **date × country** schema

### **3. Generate source-specific outbreak indicators**

* `gdelt_signal` – media-detected outbreak mentions
* `promed_signal` – ProMED alerts
* `animal_case` – FAO laboratory-confirmed animal cases
* `human_case` – WHO-reported human cases

### **4. Create a fused outbreak ground truth**

* Aggregate all signals to daily country-level
* Construct a **binary “fused_outbreak” label** reflecting activity from any source
* Provide a consistent, research-ready dataset for downstream time-series and early-warning models

### **5. Export a reproducible, transparent outbreak dataset**

* Save the final dataset as `improved_h5n1_ground_truth.csv`
* Enable future expansion to additional sources (CDC, ECDC, FluNet, etc.)

---

## **Summary**

This notebook produces a **comprehensive multi-source outbreak ground truth** that can be used to evaluate digital signals, analyze precursor behaviors, and develop early-warning models for H5N1. The method prioritizes transparency, reproducibility, and coverage across human and animal health domains.


In [ ]:
import pandas as pd
import numpy as np
import requests
from bs4 import BeautifulSoup
import feedparser
from datetime import datetime, timedelta


## Load GDELT Outbreak Signal

In [ ]:
from datetime import datetime

def parse_gdelt_date(s):
    """
    Robustly parse GDELT datetime strings.
    Handles formats like:
    - 20251010T0
    - 20240312T123000Z
    - 20250101083000
    - 20250101
    """
    s = s.strip()

    # Try removing trailing non-numeric chars
    while len(s) > 0 and not s[-1].isdigit():
        s = s[:-1]

    # Zero-pad if it ends with T0 or T
    if "T" in s:
        s = s.replace("T", "")
    
    # Now try formats by length:
    try_formats = [
        "%Y%m%d%H%M%S",
        "%Y%m%d%H%M",
        "%Y%m%d%H",
        "%Y%m%d",
    ]

    for fmt in try_formats:
        try:
            return datetime.strptime(s, fmt)
        except:
            continue

    # If all fails:
    raise ValueError(f"Unable to parse GDELT date: {s}")

def fetch_gdelt_h5n1():
    url = "https://api.gdeltproject.org/api/v2/doc/doc?query=h5n1&maxrecords=250&format=json"
    r = requests.get(url)
    data = r.json()

    rows = []
    for item in data.get("articles", []):
        try:
            dt = parse_gdelt_date(item["seendate"])
        except:
            continue
        
        rows.append({
            "date": dt,
            "country": item.get("sourcecountry", "UNK"),
            "gdelt_signal": 1
        })
        
    return pd.DataFrame(rows)


## Promed Signals

In [ ]:
def fetch_promed_h5n1():
    feed = feedparser.parse("https://promedmail.org/feed/")
    rows = []
    
    for entry in feed.entries:
        if "h5n1" in entry.title.lower() or "bird flu" in entry.title.lower():
            date = pd.to_datetime(entry.published)
            rows.append({
                "date": date,
                "country": "UNK",  # ProMED doesn't always specify country in feed
                "promed_signal": 1
            })
            
    return pd.DataFrame(rows)

promed = fetch_promed_h5n1()
print("ProMED:", promed.shape)

## FAO EMPRES-i Outbreaks (Animal)

In [ ]:
def fetch_fao_h5n1():
    url = "https://empres-i.apps.fao.org/empres-i-api/v2/event?disease=H5N1"
    r = requests.get(url)
    js = r.json()
    
    rows = []
    for ev in js:
        rows.append({
            "date": pd.to_datetime(ev["dateStart"]),
            "country": ev["country"],
            "animal_case": 1
        })
    
    return pd.DataFrame(rows)

try:
    fao = fetch_fao_h5n1()
    print("FAO EMPRES-i:", fao.shape)
except:
    fao = pd.DataFrame(columns=["date","country","animal_case"])
    print("FAO API blocked — using empty frame")

## WHO DON Scrape (HUMAN Cases)

In [ ]:
def scrape_who_don_h5n1():
    url = "https://www.who.int/emergencies/disease-outbreak-news"
    r = requests.get(url)
    soup = BeautifulSoup(r.text, "html.parser")

    rows = []
    for article in soup.find_all("a"):
        text = article.get_text().lower()
        if "h5n1" in text or "avian influenza" in text:
            # parse date from parent structure if available
            rows.append({
                "date": pd.Timestamp.today(),  # placeholder
                "country": "UNK",
                "human_case": 1
            })
    
    return pd.DataFrame(rows)

try:
    who = scrape_who_don_h5n1()
    print("WHO DON:", who.shape)
except:
    who = pd.DataFrame(columns=["date","country","human_case"])
    print("WHO scrape blocked — using empty frame")

In [ ]:
gdelt = fetch_gdelt_h5n1()

#gdelt["date"] = pd.to_datetime(gdelt["date"])  # safe now
#print("GDELT:", gdelt.shape)
#display(gdelt.head())

promed = fetch_promed_h5n1()
who = scrape_who_don_h5n1()
#fao = fetch_fao_h5n1()

## Combine All Sources and Normalize

In [ ]:
dfs = []


if not gdelt.empty: dfs.append(gdelt)
if not promed.empty: dfs.append(promed)
if not fao.empty:   dfs.append(fao)
if not who.empty:   dfs.append(who)

combined = pd.concat(dfs, ignore_index=True)

# Clean
combined["date"] = pd.to_datetime(combined["date"])
combined["country"] = combined["country"].fillna("UNK")

# Fill missing signals
combined["gdelt_signal"] = combined.get("gdelt_signal", 0)
combined["promed_signal"] = combined.get("promed_signal", 0)
combined["animal_case"] = combined.get("animal_case", 0)
combined["human_case"] = combined.get("human_case", 0)

## Aggregate to Daily Country-Level Labels

In [6]:
gt = (
    combined.groupby(["date","country"], as_index=False)
    .agg({
        "gdelt_signal":"max",
        "promed_signal":"max",
        "animal_case":"max",
        "human_case":"max"
    })
)

gt["fused_outbreak"] = (
    (gt["gdelt_signal"] +
     gt["promed_signal"] +
     gt["animal_case"] +
     gt["human_case"]) > 0
).astype(int)

print("Final GT:", gt.shape)
display(gt.head())


# ---------------------------------------------
# 7. SAVE
# ---------------------------------------------
gt.to_csv("improved_h5n1_ground_truth.csv", index=False)
print("Saved improved_h5n1_ground_truth.csv")


ProMED: (0, 0)
FAO API blocked — using empty frame
WHO DON: (0, 0)
GDELT: (250, 3)


,date,country,gdelt_signal
0,2025-10-10 00:30:00,United States,1
1,2025-10-10 00:30:00,United States,1
2,2025-10-13 19:45:00,United States,1
3,2025-10-14 08:00:00,United States,1
4,2025-09-27 02:45:00,United States,1


Final GT: (219, 7)


,date,country,gdelt_signal,promed_signal,animal_case,human_case,fused_outbreak
0,2025-09-06 00:15:00,United States,1,0,0,0,1
1,2025-09-06 01:00:00,United States,1,0,0,0,1
2,2025-09-08 03:15:00,Canada,1,0,0,0,1
3,2025-09-08 18:15:00,Morocco,1,0,0,0,1
4,2025-09-10 01:15:00,United States,1,0,0,0,1


Saved improved_h5n1_ground_truth.csv
